In [ ]:
pip install "psycopg[binary]" python-dotenv

In [1]:
import os
import psycopg
from dotenv import load_dotenv

In [3]:
# Load variables from .env into environment
load_dotenv()

DB_CONFIG = {
    "host": os.environ["DB_HOST"],
    "port": int(os.getenv("DB_PORT", 5432)),
    "dbname": os.environ["DB_NAME"],
    "user": os.environ["DB_USER"],
    "password": os.environ["DB_PASSWORD"],
}

CSV_FILE_PATH = os.getenv("CSV_FILE_PATH", "../../data/boku_waldbrand_02-11-2026.csv")

In [5]:
def load_csv_to_postgres(csv_path: str, db_config: dict) -> None:
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV-file not found: {csv_path}")
    conn = psycopg.connect(**db_config) 
    try:
        with conn:
            with conn.cursor() as cur:
            # temporary staging table `raw_csv`
                cur.execute("""
                CREATE TEMP TABLE raw_csv (
                    objid INT,
                    waldbrandid BIGINT,
                    jahr INT,
                    monat INT,
                    tag INT,
                    bundesland TEXT,
                    gemeinde TEXT,
                    brandart INT,
                    brandflaeche NUMERIC,
                    ursache INT,
                    feuerwehren INT,
                    einsatzkraefte INT,
                    quelle1 TEXT,
                    verifiziert BOOLEAN,
                    gemeinde_neu TEXT,
                    brandflaeche_korr NUMERIC,
                    schutzwald INT,
                    longitude DOUBLE PRECISION,
                    latitude DOUBLE PRECISION
                ) ON COMMIT DROP;
                """)

                # stream CSV into raw_csv (maps ursache1 to ursache)
                copy_sql = """
                COPY raw_csv (
                    objid, waldbrandid, jahr, monat, tag, bundesland, gemeinde,
                    brandart, brandflaeche, ursache, feuerwehren, einsatzkraefte,
                    quelle1, verifiziert, gemeinde_neu, brandflaeche_korr,
                    schutzwald, longitude, latitude
                )
                FROM STDIN WITH (FORMAT CSV, HEADER TRUE, DELIMITER ',', NULL '', ENCODING 'UTF8');
                """
                with open(csv_path, "r", encoding="utf-8") as f:
                    with cur.copy(copy_sql) as copy:
                        while chunk := f.read(65536):
                            copy.write(chunk)

                # INSERT 
                upsert_sql = """
                INSERT INTO public.wildfires (
                    objid, waldbrandid, brand_date, jahr, monat, tag, gemeinde_id, brandart_id, ursache_id,
                    brandflaeche, brandflaeche_korr, feuerwehren, einsatzkraefte, quelle1, verifiziert,
                    schutzwald, longitude, latitude
                )
                SELECT 
                    r.objid,
                    r.waldbrandid,
                    make_date(r.jahr, r.monat, r.tag) AS brand_date,
                    r.jahr,
                    r.monat,
                    r.tag,
                    g.gemeinde_id,
                    NULLIF(r.brandart, -9999) AS brandart_id,
                    NULLIF(r.ursache, -9999) AS ursache_id,
                    NULLIF(r.brandflaeche, -9999) AS brandflaeche,
                    NULLIF(r.brandflaeche_korr, -9999) AS brandflaeche_korr,
                    NULLIF(r.feuerwehren, -9999) AS feuerwehren,
                    NULLIF(r.einsatzkraefte, -9999) AS einsatzkraefte,
                    r.quelle1,
                    r.verifiziert,
                    NULLIF(r.schutzwald, -9999)::SMALLINT AS schutzwald,
                    NULLIF(r.longitude, -9999) AS longitude,
                    NULLIF(r.latitude, -9999) AS latitude
                FROM raw_csv r
                LEFT JOIN public.gemeinde g 
                    ON g.name = COALESCE(NULLIF(TRIM(r.gemeinde_neu), ''), TRIM(r.gemeinde))
                    AND g.bundesland_code = TRIM(r.bundesland)
                ON CONFLICT (objid) DO UPDATE SET
                    waldbrandid = EXCLUDED.waldbrandid,
                    brand_date = EXCLUDED.brand_date,
                    jahr = EXCLUDED.jahr,
                    monat = EXCLUDED.monat,
                    tag = EXCLUDED.tag,
                    gemeinde_id = EXCLUDED.gemeinde_id,
                    brandart_id = EXCLUDED.brandart_id,
                    ursache_id = EXCLUDED.ursache_id,
                    brandflaeche = EXCLUDED.brandflaeche,
                    brandflaeche_korr = EXCLUDED.brandflaeche_korr,
                    feuerwehren = EXCLUDED.feuerwehren,
                    einsatzkraefte = EXCLUDED.einsatzkraefte,
                    quelle1 = EXCLUDED.quelle1,
                    verifiziert = EXCLUDED.verifiziert,
                    schutzwald = EXCLUDED.schutzwald,
                    longitude = EXCLUDED.longitude,
                    latitude = EXCLUDED.latitude;
                """
                cur.execute(upsert_sql)
                print(f"Successfully upserted {cur.rowcount} rows into public.wildfires.")

                # Explicitly drop temporary table after successful insert
                cur.execute("DROP TABLE IF EXISTS raw_csv;")
                print("Temporary table raw_csv dropped.")

    finally:
        conn.close()


In [6]:
load_csv_to_postgres(CSV_FILE_PATH, DB_CONFIG)

Successfully upserted 6679 rows into public.wildfires.
Temporary table raw_csv dropped.


### TO-DO

In [ ]:
import os
import psycopg2
from dotenv import load_dotenv

load_dotenv()

DB_CONFIG = {
    "host": os.environ["DB_HOST"],
    "port": int(os.getenv("DB_PORT", 5432)),
    "dbname": os.environ["DB_NAME"],
    "user": os.environ["DB_USER"],
    "password": os.environ["DB_PASSWORD"],
}

REQUESTS_SQL_PATH = os.getenv("REQUESTS_SQL_PATH", "generate_requests.sql")

def run_pipeline(requests_sql_path: str, db_config: dict) -> None:
    if not os.path.exists(requests_sql_path):
        raise FileNotFoundError(f"SQL file not found: {requests_sql_path}")

    with open(requests_sql_path, "r", encoding="utf-8") as f:
        generate_requests_sql = f.read()

    conn = psycopg2.connect(**db_config)
    try:
        # Phase 1: Transactional CSV Load + Cleanup
        with conn:
            with conn.cursor() as cur:
                cur.execute("""  """)
                        
            with conn.cursor() as cur:
            # Phase 2: Refresh Materialized View
            cur.execute("REFRESH MATERIALIZED VIEW public.mv_wildfire_suitable_control_dates;")
            print("Refreshed materialized view public.mv_wildfire_suitable_control_dates.")

            # Phase 3: Execute SQL script to generate the requests table
            cur.execute(generate_requests_sql)
            print(f"Executed {requests_sql_path} successfully.")

    finally:
        conn.close()


if __name__ == "__main__":
    run_pipeline(REQUESTS_SQL_PATH, DB_CONFIG)